# I. Import
Exploration initiale

In [5]:
# Imports

import sys
sys.executable
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 10)

## Déinir les constantes

In [6]:
# Définir les constantes

DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'


## 1.1 Charger le dataset

In [7]:
# Chargement de la dataset
def loadData(path = DATA_PATH):
    df_raw = pd.read_csv(path)

    ## Premier aperçut de la forme et Vérification des colonnes
    print(f'Shape : {df_raw.shape}')
    print(f'\nColonnes ({len(df_raw.columns)}) :')
    print(df_raw.columns.tolist())

    return df_raw

df_raw = loadData()

Shape : (7043, 21)

Colonnes (21) :
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [8]:
# Vue d'ensemble
df_raw.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1.2 diagnostic essentiels

In [9]:
# types de données
print('Types de données')
df_raw.info()

Types de données
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   st

In [10]:
# Valeurs manquantes
missing = df_raw.isnull().sum()

if missing.sum() > 0:
    print(missing[missing > 0])
print("Aucune valeur manquante détectée")

Aucune valeur manquante détectée


### Statistiques descriptives

In [ ]:
# Uniquement les colonnes numériques : génère un résumé statistique des colonnes numériques du Dataframe
df_raw.describe().round(2)

,SeniorCitizen,tenure,MonthlyCharges
count,7043.00,7043.00,7043.00
mean,0.16,32.37,64.76
std,0.37,24.56,30.09
min,0.00,0.00,18.25
25%,0.00,9.00,35.50
50%,0.00,29.00,70.35
75%,0.00,55.00,89.85
max,1.00,72.00,118.75


In [17]:
# Exploration catégorielle des données non numérique
cat_cols = df_raw.select_dtypes(include='object').columns.to_list()
print(f'Colonnes catégorielle: {len(cat_cols)} \n')

for col in cat_cols:
    print(f'{col:25s} -> {df_raw[col].nunique()} valeurs: {df_raw[col].unique()[:5].tolist()}')

Colonnes catégorielle: 18 

customerID                -> 7043 valeurs: ['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU']
gender                    -> 2 valeurs: ['Female', 'Male']
Partner                   -> 2 valeurs: ['Yes', 'No']
Dependents                -> 2 valeurs: ['No', 'Yes']
PhoneService              -> 2 valeurs: ['No', 'Yes']
MultipleLines             -> 3 valeurs: ['No phone service', 'No', 'Yes']
InternetService           -> 3 valeurs: ['DSL', 'Fiber optic', 'No']
OnlineSecurity            -> 3 valeurs: ['No', 'Yes', 'No internet service']
OnlineBackup              -> 3 valeurs: ['Yes', 'No', 'No internet service']
DeviceProtection          -> 3 valeurs: ['No', 'Yes', 'No internet service']
TechSupport               -> 3 valeurs: ['No', 'Yes', 'No internet service']
StreamingTV               -> 3 valeurs: ['No', 'Yes', 'No internet service']
StreamingMovies           -> 3 valeurs: ['No', 'Yes', 'No internet service']
Contract                  -> 3 v

## 1.3 Analyser la variable cible

In [23]:
# Variable cible : Churn
churn_count = df_raw['Churn'].value_counts()
churn_pct = df_raw['Churn'].value_counts(normalize=True).round(4)*100

print('Distribution du Churn')
for val in churn_count.index:
    print(f'{val:5s} : {churn_count[val]:5d} clients ({churn_pct[val]:.1f}%)')

print(f'Classes déséquilibrées : {churn_pct['Yes']:.1f}% de churners')

Distribution du Churn
No    :  5174 clients (73.5%)
Yes   :  1869 clients (26.5%)
Classes déséquilibrées : 26.5% de churners


On utilisera AUC et F1-score comme métriques principales (pas l'accuracy)

#### Vérification critique : Total Charges

In [35]:
# TotalCharges est de type 'object' alors qu'il devrait être float

print(f'Type de TotalCharges : {df_raw['TotalCharges'].dtype}')
print('\nValeurs non-numériques :')
non_numeric = df_raw[pd.to_numeric(df_raw['TotalCharges'], errors='coerce').isna()]
print(f'{len(non_numeric)} lignes contiennent des espaces au lieu des valeurs numériques.\n')
print(non_numeric[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head())

Type de TotalCharges : str

Valeurs non-numériques :
11 lignes contiennent des espaces au lieu des valeurs numériques.

      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             


Les clients en tenure =0 : ils viennent d'arriver
TotalCharges n'a pas encore été calculé -> valeur vide stockée comme ' '